In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [ ]:
trades = pd.read_csv(r"C:\Users\ASUS\Downloads\historical_data.csv")
fear = pd.read_csv(r"C:\Users\ASUS\Downloads\fear_greed_index.csv")

print("Trades Dataset Shape:", trades.shape)
print("Fear & Greed Dataset Shape:", fear.shape)

display(trades.head())
display(fear.head())

In [ ]:
print("Trades Dataset Information")
trades.info()

print("\nFear & Greed Dataset Information")
fear.info()

print("\nMissing Values in Trades Dataset")
display(trades.isnull().sum())

print("\nMissing Values in Fear & Greed Dataset")
display(fear.isnull().sum())

print("\nDuplicate Records")
print("Trades:", trades.duplicated().sum())
print("Fear & Greed:", fear.duplicated().sum())

In [ ]:
trades["Timestamp IST"] = pd.to_datetime(
    trades["Timestamp IST"],
    dayfirst=True
)

fear["date"] = pd.to_datetime(fear["date"])

trades["Date"] = trades["Timestamp IST"].dt.date
fear["Date"] = fear["date"].dt.date

trades.drop_duplicates(inplace=True)
fear.drop_duplicates(inplace=True)

numeric_columns = [
    "Closed PnL",
    "Fee",
    "Size USD",
    "Execution Price",
    "Size Tokens"
]

for column in numeric_columns:
    trades[column] = pd.to_numeric(
        trades[column],
        errors="coerce"
    )

print("Data cleaning completed successfully.")

In [ ]:
merged = pd.merge(
    trades,
    fear[["Date", "classification", "value"]],
    on="Date",
    how="left"
)

merged.rename(
    columns={
        "classification": "Market Sentiment",
        "value": "Fear & Greed Index"
    },
    inplace=True
)

print("Merged Dataset Shape:", merged.shape)

display(merged.head())

In [ ]:
print("Merged Dataset Information")
merged.info()

print("\nMissing Values After Merge")
display(merged.isnull().sum())

print("\nMarket Sentiment Distribution")
display(merged["Market Sentiment"].value_counts())

In [ ]:
print("Summary Statistics")

display(
    merged[
        [
            "Execution Price",
            "Size Tokens",
            "Size USD",
            "Fee",
            "Closed PnL",
            "Fear & Greed Index"
        ]
    ].describe()
)

In [ ]:
plt.figure(figsize=(8,5))

merged["Market Sentiment"].value_counts().plot(
    kind="bar"
)

plt.title("Distribution of Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Number of Trades")

plt.xticks(rotation=30)

plt.show()

In [ ]:
avg_pnl = (
    merged.groupby("Market Sentiment")["Closed PnL"]
    .mean()
    .sort_values()
)

plt.figure(figsize=(8,5))

avg_pnl.plot(kind="bar")

plt.title("Average Closed PnL by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average Closed PnL")

plt.xticks(rotation=30)

plt.show()

display(avg_pnl)

In [ ]:
trade_count = merged.groupby(
    "Market Sentiment"
).size()

plt.figure(figsize=(8,5))

trade_count.plot(kind="bar")

plt.title("Trading Activity Across Market Sentiments")
plt.xlabel("Market Sentiment")
plt.ylabel("Number of Trades")

plt.xticks(rotation=30)

plt.show()

display(trade_count)

In [ ]:
side = merged["Side"].value_counts()

plt.figure(figsize=(6,6))

plt.pie(
    side,
    labels=side.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Buy vs Sell Distribution")

plt.show()

In [ ]:
trade_size = (
    merged.groupby("Market Sentiment")["Size USD"]
    .mean()
)

plt.figure(figsize=(8,5))

trade_size.plot(kind="bar")

plt.title("Average Trade Size by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average Trade Size (USD)")

plt.xticks(rotation=30)

plt.show()

display(trade_size)

In [ ]:
fees = (
    merged.groupby("Market Sentiment")["Fee"]
    .mean()
)

plt.figure(figsize=(8,5))

fees.plot(kind="bar")

plt.title("Average Trading Fee by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Average Fee")

plt.xticks(rotation=30)

plt.show()

display(fees)

In [ ]:
daily_pnl = merged.groupby(
    "Date"
)["Closed PnL"].sum()

plt.figure(figsize=(14,5))

plt.plot(
    daily_pnl.index,
    daily_pnl.values
)

plt.title("Daily Closed PnL")

plt.xlabel("Date")
plt.ylabel("Closed PnL")

plt.xticks(rotation=45)

plt.show()

In [ ]:
merged["Winning Trade"] = np.where(
    merged["Closed PnL"] > 0,
    1,
    0
)

win_rate = (
    merged.groupby("Market Sentiment")["Winning Trade"]
    .mean()
    * 100
).sort_values()

plt.figure(figsize=(8,5))

win_rate.plot(kind="bar")

plt.title("Win Rate by Market Sentiment")
plt.xlabel("Market Sentiment")
plt.ylabel("Win Rate (%)")

plt.xticks(rotation=30)

plt.show()

display(win_rate)

In [ ]:
corr = merged[
    [
        "Execution Price",
        "Size Tokens",
        "Size USD",
        "Fee",
        "Closed PnL",
        "Fear & Greed Index"
    ]
].corr()

display(corr)

plt.figure(figsize=(8,6))

plt.imshow(corr, cmap="coolwarm")

plt.colorbar()

plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)

plt.title("Correlation Matrix")

plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

features = merged[
    [
        "Size USD",
        "Fee",
        "Closed PnL",
        "Fear & Greed Index"
    ]
].fillna(0)

scaler = StandardScaler()

scaled = scaler.fit_transform(features)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

merged["Cluster"] = kmeans.fit_predict(scaled)

cluster_summary = merged.groupby("Cluster")[
    [
        "Size USD",
        "Closed PnL",
        "Fee"
    ]
].mean()

display(cluster_summary)

In [ ]:
cluster_count = merged["Cluster"].value_counts()

plt.figure(figsize=(6,5))

cluster_count.plot(kind="bar")

plt.title("Trader Clusters")

plt.xlabel("Cluster")

plt.ylabel("Number of Trades")

plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X = merged[
    [
        "Execution Price",
        "Size Tokens",
        "Size USD",
        "Fee",
        "Fear & Greed Index"
    ]
].fillna(0)

y = merged["Closed PnL"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

prediction = model.predict(X_test)

print("R² Score :", r2_score(y_test, prediction))

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values()

plt.figure(figsize=(8,5))

importance.plot(kind="barh")

plt.title("Feature Importance")

plt.show()

display(importance)

In [ ]:
print("Key Insights")

print("""
1. Trading performance varies across different market sentiment conditions.

2. Win rate changes noticeably during Fear and Greed phases.

3. Trade size has a stronger influence on profitability than transaction fees.

4. Trader segmentation identified different trading behaviors based on trade size and profitability.

5. Random Forest results indicate that trade size and execution price contribute significantly to predicting Closed PnL.
""")

In [ ]:
print("Recommendations")

print("""
1. Reduce position size during Extreme Fear periods to manage downside risk.

2. Increase exposure only when historical performance during Greed periods remains consistently positive.

3. Monitor high-value trades more closely, as trade size has the greatest impact on profitability.

4. Use market sentiment as an additional feature in risk management and trading strategies.

5. Segment traders based on historical performance to develop personalized trading strategies.
""")